In [1]:
from langchain_community.document_loaders import TextLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_ollama import OllamaEmbeddings

from langchain_community.vectorstores import FAISS

from langchain_community.retrievers import BM25Retriever

from langchain_classic.retrievers import EnsembleRetriever

C:\Users\gagan\AppData\Local\Temp\ipykernel_5768\2403852858.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
c:\Users\gagan\Desktop\Langchain\labs\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dummy_text = """
Chunk 1:
To fix the server,
check Error Code XYZ-789
in the main terminal.

Chunk 2:
The company policy requires
a 30-day notice period.

Chunk 3:
System failures and bugs
can often be fixed
by restarting the router.

Chunk 4:
Employee ID XYZ-789
belongs to John Doe
from the IT department.
"""

with open("hybrid_test.txt", "w", encoding="utf-8") as f:
    f.write(dummy_text)

In [3]:
loader = TextLoader("hybrid_test.txt")

docs = loader.load()

In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=0
)

splits = splitter.split_documents(docs)

print("Chunks:", len(splits))

Chunks: 4


In [5]:
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

In [6]:
vectorstore = FAISS.from_documents(
    splits,
    embeddings
)

dense_retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 2
    }
)

In [7]:
sparse_retriever = BM25Retriever.from_documents(
    splits
)

sparse_retriever.k = 2

In [8]:
hybrid_retriever = EnsembleRetriever(
    retrievers=[
        sparse_retriever,
        dense_retriever
    ],
    weights=[
        0.5,
        0.5
    ]
)

In [9]:
query = "How to handle Error Code XYZ-789?"

print(query)

How to handle Error Code XYZ-789?


In [10]:
print("=" * 80)
print("DENSE RETRIEVER")
print("=" * 80)

for doc in dense_retriever.invoke(query):
    print(doc.page_content)
    print("-" * 50)

DENSE RETRIEVER
Chunk 1:
To fix the server,
check Error Code XYZ-789
in the main terminal.
--------------------------------------------------
Chunk 4:
Employee ID XYZ-789
belongs to John Doe
from the IT department.
--------------------------------------------------


In [11]:
print("=" * 80)
print("BM25 RETRIEVER")
print("=" * 80)

for doc in sparse_retriever.invoke(query):
    print(doc.page_content)
    print("-" * 50)

BM25 RETRIEVER
Chunk 1:
To fix the server,
check Error Code XYZ-789
in the main terminal.
--------------------------------------------------
Chunk 4:
Employee ID XYZ-789
belongs to John Doe
from the IT department.
--------------------------------------------------


In [12]:
print("=" * 80)
print("HYBRID RETRIEVER")
print("=" * 80)

for doc in hybrid_retriever.invoke(query):
    print(doc.page_content)
    print("-" * 50)

HYBRID RETRIEVER
Chunk 1:
To fix the server,
check Error Code XYZ-789
in the main terminal.
--------------------------------------------------
Chunk 4:
Employee ID XYZ-789
belongs to John Doe
from the IT department.
--------------------------------------------------
